# SupremeAI - AirLLM Inference Server

Runs AirLLM on Colab T4 GPU and serves OpenAI-compatible API for SupremeAI.

### Requirements:
1. Runtime > Change runtime type > T4 GPU
2. HuggingFace Token - https://huggingface.co/settings/tokens
3. ngrok Token - https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Step 1: Install dependencies (with version compatibility)
!pip uninstall -y transformers optimum accelerate bitsandbytes airllm -q 2>/dev/null
!pip install -q transformers accelerate bitsandbytes
!pip install -q optimum  # Install without bettertransformer extra
!pip install -q airllm
!pip install -q flask flask-cors pyngrok

import torch
import sys
print('Installed packages:')
print(f'  Python: {sys.version.split()[0]}')
print(f'  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ NO GPU - Change Runtime to T4!')

In [ ]:
# Step 2: Configure tokens and model
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'
MODEL_ID = 'meta-llama/Llama-3.3-70B-Instruct'
COMPRESSION = '4bit'
MAX_NEW_TOKENS = 512
PORT = 8081
print('Model:', MODEL_ID, '| Compression:', COMPRESSION)

In [ ]:
# Step 3: Load model
import time
import traceback

# Load model with AirLLM (optimized)
print('Loading model...')
start = time.time()

try:
    from airllm import AutoModel
    kw = {'compression': COMPRESSION}
    if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 10:
        kw['hf_token'] = HF_TOKEN
    model = AutoModel.from_pretrained(MODEL_ID, **kw)
    print(f'✓ Model loaded in {time.time() - start:.0f}s')
except Exception as e:
    print(f'✗ Failed with AirLLM: {str(e)}')
    print('  Falling back to standard transformers...')
    
    # Fallback: Load directly with transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN if HF_TOKEN.startswith('hf_') else None,
        device_map='auto',
        load_in_4bit=True if COMPRESSION == '4bit' else False
    )
    print(f'✓ Model loaded (transformers) in {time.time() - start:.0f}s')

In [ ]:
# Step 4: Start API Server + ngrok
import uuid, time, torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)
stats = {'req': 0, 'err': 0, 'start': time.time()}

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status':'healthy','provider':'airllm-colab','model':MODEL_ID,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu','uptime':int(time.time()-stats['start']),'requests':stats['req']})

@app.route('/v1/models', methods=['GET'])
def list_models():
    return jsonify({'object':'list','data':[{'id':MODEL_ID,'object':'model','owned_by':'airllm-colab'}]})

@app.route('/v1/chat/completions', methods=['POST'])
def chat():
    try:
        stats['req'] += 1
        data = request.json
        msgs = data.get('messages', [])
        mt = min(data.get('max_tokens', MAX_NEW_TOKENS), MAX_NEW_TOKENS)
        parts = []
        for m in msgs:
            r, c = m.get('role','user'), m.get('content','')
            if r == 'system': parts.append('[INST] <<SYS>>\n'+c+'\n<</SYS>> [/INST]')
            elif r == 'user': parts.append('[INST] '+c+' [/INST]')
            elif r == 'assistant': parts.append(c)
        prompt = '\n'.join(parts)
        toks = model.tokenizer([prompt], return_tensors='pt', return_attention_mask=False, truncation=True, max_length=2048, padding=False)
        il = len(toks['input_ids'][0])
        gen = model.generate(toks['input_ids'].cuda(), max_new_tokens=mt, use_cache=True, return_dict_in_generate=True)
        nt = gen.sequences[0][il:]
        txt = model.tokenizer.decode(nt, skip_special_tokens=True).strip()
        return jsonify({'id':'chatcmpl-'+uuid.uuid4().hex[:12],'object':'chat.completion','created':int(time.time()),'model':data.get('model',MODEL_ID),'choices':[{'index':0,'message':{'role':'assistant','content':txt},'finish_reason':'stop'}],'usage':{'prompt_tokens':il,'completion_tokens':len(nt),'total_tokens':il+len(nt)}})
    except Exception as e:
        stats['err'] += 1
        return jsonify({'error':{'message':str(e),'type':'server_error','code':500}}), 500

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT)
url = tunnel.public_url
print('\n' + '='*60)
print('SupremeAI AirLLM Server LIVE!')
print('='*60)
print('Public URL:    ' + url)
print('Chat Endpoint: ' + url + '/v1/chat/completions')
print('Health Check:  ' + url + '/health')
print('='*60)
print('Set in SupremeAI:')
print('  AIRLLM_ENDPOINT=' + url + '/v1/chat/completions')
print('  AIRLLM_HEALTHCHECK_URL=' + url + '/health')
print('='*60)
print('Cell running - do not stop!')
app.run(host='0.0.0.0', port=PORT)